# 06 — Model 2: Hybrid (AlephBERT + ResNet50)

The original proposal's contingency plan, built now that three SigLIP2 strategies (frozen/unfreeze/LoRA) plus a cross-attention fusion upgrade all failed to beat `text_only_bert` (0.704 ROC-AUC / 0.212 PR-AUC). See `notebooks/04_model1_crossattn_training.ipynb` and `05_model1b_crossattn_lora_only.ipynb` for that story.

**Completely different architecture** from SigLIP2 -- two backbones that were never jointly pretrained (unlike SigLIP2's contrastively-aligned text/image towers):
- **Text**: AlephBERT (Hebrew-specific, not SigLIP2's multilingual-but-English-heavy encoder)
- **Image**: ResNet50 (ImageNet-pretrained)
- **Fusion**: concat + small MLP (not cross-attention -- exp05 showed that didn't help when isolated, and here the two backbones don't share an embedding space to begin with)

Conservative unfreeze by default (top 1 BERT layer + ResNet's layer4) -- Stage B's lesson was that too many trainable params on ~87K rows overfits fast. 22.8M trainable params (15.2%), verified via smoke test (46 backbone tensors + 5 head tensors, all gradients flowing correctly) before this real run.

Run via `configs/exp06_model2_hybrid.yaml` -- full dataset (not the full-res-only filter from Model 1).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run training

**Do not run this at the same time as any other GPU/MPS process** -- running two MPS-heavy processes concurrently has caused real hangs (multiple times now) needing a full machine restart. Run one at a time.

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp06_model2_hybrid.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/hybrid_model2_alephbert_resnet50"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation epochs logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Model 2 (Hybrid) training loss")

axes[1].plot(val_log["epoch"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["epoch"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.212, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.212)")
axes[1].axhline(0.184, color="green", linestyle=":", label="Stage C (best SigLIP2) PR-AUC (0.184)")
axes[1].set_xlabel("epoch")
axes[1].set_title("Model 2 (Hybrid) validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

In [ ]:
best_hybrid = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208},
    {"model": "text_only_bert",       "roc_auc": 0.704, "pr_auc": 0.212},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167},
    {"model": "siglip2_stage_c_lora", "roc_auc": 0.659, "pr_auc": 0.184},
    {"model": "model1_crossattn_full","roc_auc": 0.625, "pr_auc": 0.161},
    {"model": "model1b_crossattn_lora_only", "roc_auc": 0.643, "pr_auc": 0.179},
    {"model": "model2_hybrid",        "roc_auc": best_hybrid["roc_auc"], "pr_auc": best_hybrid["pr_auc"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
